In [ ]:
"""
Google ISLR (asl-signs) - PyTorch version. Run it directly on Kaggle (cell or script):
it preprocesses the landmark files once (cached), then trains and saves /kaggle/working/model.pt.
Edit the config block below if you want different settings.
"""
import json
import math
import os
import time
import warnings
from multiprocessing import Pool

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

# ----------------------------------------------------------------------------
# Config
# ----------------------------------------------------------------------------
SEED = 42
INPUT_SIZE = 64          # frames per sample after resampling
NUM_CLASSES = 250
N_ROWS, N_DIMS = 543, 3  # landmarks per frame, (x, y, z)

# Run config (no CLI arguments)
DATA_DIR = "/kaggle/input/competitions/asl-signs"   # has train.csv, train_landmark_files/, sign_to_prediction_index_map.json
CACHE_DIR = "/kaggle/working/cache"
OUT_PATH = "/kaggle/working/model.pt"
WORKERS = os.cpu_count() or 2
LIMIT = 0              # debug: use only N random samples (0 = all)
USE_VAL = True         # True: hold out 10% of participants and report val metrics
EPOCHS = 100
LR = 1e-3
WD_RATIO = 0.05
ALL_SIGNS_N = 4        # samples per class per batch (batch = 250 * 4 = 1000)
USE_AMP = True
DATA_ON_GPU = True     # keep the whole dataset (~4.4GB) on GPU; set False if you get OOM

# Model
UNITS = 512              # transformer width
LANDMARK_UNITS = 384     # per-landmark-group embedding width
NUM_BLOCKS = 2
NUM_HEADS = 8
MLP_RATIO = 2
MLP_DROPOUT = 0.30
CLASSIFIER_DROPOUT = 0.10
LABEL_SMOOTHING = 0.25
FRAME_MASK_P = 0.25      # random frame dropout during training

# Augmentation (training only, applied on GPU to raw x/y before normalisation)
AUGMENT = True
AUG_P = 0.8              # fraction of samples in a batch that get augmented
AUG_SCALE = (0.8, 1.2)   # independent x / y scale -> simulates other camera aspect ratio / distance
AUG_ROT_DEG = 15         # random rotation around the signer's center
AUG_SHIFT = 0.08         # random translation (coordinates are in [0, 1])
AUG_NOISE = 0.004        # per-landmark gaussian jitter
AUG_LIPS_DROP_P = 0.10   # drop the whole lips group for some samples

# ----------------------------------------------------------------------------
# Landmark indices
# ----------------------------------------------------------------------------
LIPS0 = np.array([
    61, 185, 40, 39, 37, 0, 267, 269, 270, 409,
    291, 146, 91, 181, 84, 17, 314, 405, 321, 375,
    78, 191, 80, 81, 82, 13, 312, 311, 310, 415,
    95, 88, 178, 87, 14, 317, 402, 318, 324, 308,
])
LEFT_HAND0 = np.arange(468, 489)
RIGHT_HAND0 = np.arange(522, 543)
LEFT_POSE0 = np.array([502, 504, 506, 508, 510])
RIGHT_POSE0 = np.array([503, 505, 507, 509, 511])
LM_LEFT = np.concatenate([LIPS0, LEFT_HAND0, LEFT_POSE0])    # left-dominant layout
LM_RIGHT = np.concatenate([LIPS0, RIGHT_HAND0, RIGHT_POSE0])  # right-dominant layout
N_COLS = len(LM_LEFT)  # 66 = 40 lips + 21 hand + 5 pose
LIPS_SL, HAND_SL, POSE_SL = slice(0, 40), slice(40, 61), slice(61, 66)


# ----------------------------------------------------------------------------
# Preprocessing (numpy version of the TF PreprocessLayer)
# ----------------------------------------------------------------------------
def load_parquet(path):
    df = pd.read_parquet(path, columns=["x", "y", "z"])
    n_frames = len(df) // N_ROWS
    return df.values.reshape(n_frames, N_ROWS, N_DIMS).astype(np.float32)


def _pad_edge(a, left, right):
    return np.concatenate([np.repeat(a[:1], left, axis=0), a, np.repeat(a[-1:], right, axis=0)], axis=0)


def preprocess_sequence(raw):
    """raw: [T, 543, 3] (NaN = missing) -> data [64, 66, 3], frame_idxs [64] (-1 = padding)."""
    # Dominant hand = the hand with more non-NaN values. Everything is mapped to a "left" hand.
    left_cnt = (~np.isnan(raw[:, LEFT_HAND0])).sum()
    right_cnt = (~np.isnan(raw[:, RIGHT_HAND0])).sum()
    left_dom = left_cnt >= right_cnt
    hand = raw[:, LEFT_HAND0] if left_dom else raw[:, RIGHT_HAND0]
    keep = np.flatnonzero((~np.isnan(hand)).sum(axis=(1, 2)) > 0)  # frames where the hand exists

    if len(keep) == 0:  # no hands at all -> a single empty frame
        data = np.zeros((1, N_COLS, N_DIMS), dtype=np.float32)
        fidx = np.zeros(1, dtype=np.float32)
    else:
        data = raw[np.ix_(keep, LM_LEFT if left_dom else LM_RIGHT)].copy()
        fidx = (keep - keep.min()).astype(np.float32)
        if not left_dom:  # mirror x of hand + pose so right-handed signers look left-handed
            data[:, HAND_SL.start:, 0] = 1.0 - data[:, HAND_SL.start:, 0]

    n = len(data)
    if n < INPUT_SIZE:  # pad
        out = np.zeros((INPUT_SIZE, N_COLS, N_DIMS), dtype=np.float32)
        out[:n] = np.nan_to_num(data, nan=0.0)
        idx = np.full(INPUT_SIZE, -1, dtype=np.float32)
        idx[:n] = fidx
        return out, idx

    # too long (or exactly right) -> repeat, edge-pad and mean-pool down to INPUT_SIZE
    if n < INPUT_SIZE ** 2:
        rep = INPUT_SIZE * INPUT_SIZE // n
        data = np.repeat(data, rep, axis=0)
        fidx = np.repeat(fidx, rep, axis=0)
    n = len(data)
    pool = -(-n // INPUT_SIZE)  # ceil
    pad = pool * INPUT_SIZE - n
    pad_l = pad // 2 + INPUT_SIZE // 2
    pad_r = pad // 2 + INPUT_SIZE // 2 + (pad % 2)
    data = _pad_edge(data, pad_l, pad_r).reshape(INPUT_SIZE, -1, N_COLS, N_DIMS)
    fidx = _pad_edge(fidx, pad_l, pad_r).reshape(INPUT_SIZE, -1)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)  # all-NaN slices
        data = np.nanmean(data, axis=1)
    fidx = fidx.mean(axis=1)
    return np.nan_to_num(data, nan=0.0).astype(np.float32), fidx.astype(np.float32)


def _work(path):
    return preprocess_sequence(load_parquet(path))


def preprocess_all():
    os.makedirs(CACHE_DIR, exist_ok=True)
    train = pd.read_csv(f"{DATA_DIR}/train.csv")
    if LIMIT:
        train = train.sample(LIMIT, random_state=SEED).reset_index(drop=True)
    with open(f"{DATA_DIR}/sign_to_prediction_index_map.json") as f:
        sign2idx = json.load(f)
    y = train["sign"].map(sign2idx).values.astype(np.int64)
    paths = [f"{DATA_DIR}/{p}" for p in train["path"]]
    n = len(paths)

    X = np.lib.format.open_memmap(f"{CACHE_DIR}/X.npy", mode="w+", dtype=np.float32,
                                  shape=(n, INPUT_SIZE, N_COLS, N_DIMS))
    FI = np.lib.format.open_memmap(f"{CACHE_DIR}/fidx.npy", mode="w+", dtype=np.float32,
                                   shape=(n, INPUT_SIZE))
    with Pool(max(1, WORKERS)) as pool:
        for i, (d, f) in enumerate(tqdm(pool.imap(_work, paths, chunksize=32), total=n)):
            X[i], FI[i] = d, f
    X.flush()
    FI.flush()
    np.save(f"{CACHE_DIR}/y.npy", y)
    np.save(f"{CACHE_DIR}/participant.npy", train["participant_id"].values)
    with open(f"{CACHE_DIR}/sign2idx.json", "w") as f:
        json.dump(sign2idx, f)
    print(f"done: X {X.shape}, saved to {CACHE_DIR}")


# ----------------------------------------------------------------------------
# Normalisation stats (mean/std of non-zero x,y per landmark)
# ----------------------------------------------------------------------------
def compute_stats(X, idxs, bs=1024):
    sls = {"lips": LIPS_SL, "hand": HAND_SL, "pose": POSE_SL}
    s = {k: np.zeros((v.stop - v.start, 2)) for k, v in sls.items()}
    s2 = {k: np.zeros_like(v) for k, v in s.items()}
    c = {k: np.zeros_like(v) for k, v in s.items()}
    for i in tqdm(range(0, len(idxs), bs), desc="stats"):
        chunk = X[idxs[i:i + bs]][..., :2].astype(np.float64)
        for k, sl in sls.items():
            v = chunk[:, :, sl]
            m = v != 0
            s[k] += (v * m).sum((0, 1))
            s2[k] += (v * v * m).sum((0, 1))
            c[k] += m.sum((0, 1))
    stats = {}
    for k in sls:
        cnt = np.maximum(c[k], 1)
        mean = s[k] / cnt
        std = np.sqrt(np.maximum(s2[k] / cnt - mean ** 2, 1e-12))
        stats[f"{k}_mean"] = torch.tensor(mean, dtype=torch.float32)
        stats[f"{k}_std"] = torch.tensor(std, dtype=torch.float32)
    return stats


# ----------------------------------------------------------------------------
# Model
# ----------------------------------------------------------------------------
def glorot(layer):
    nn.init.xavier_uniform_(layer.weight)
    if layer.bias is not None:
        nn.init.zeros_(layer.bias)
    return layer


def he_uniform(layer):
    nn.init.kaiming_uniform_(layer.weight, nonlinearity="relu")
    if layer.bias is not None:
        nn.init.zeros_(layer.bias)
    return layer


class LandmarkEmbedding(nn.Module):
    """MLP embedding per frame; frames where the landmark group is missing get a learned 'empty' vector."""

    def __init__(self, in_dim, units):
        super().__init__()
        self.empty = nn.Parameter(torch.zeros(units))
        self.fc1 = glorot(nn.Linear(in_dim, units, bias=False))
        self.fc2 = he_uniform(nn.Linear(units, units, bias=False))

    def forward(self, x):  # x [B, T, in_dim]
        e = self.fc2(F.gelu(self.fc1(x)))
        missing = (x == 0).all(dim=-1, keepdim=True)
        return torch.where(missing, self.empty.to(e.dtype), e)


class Embedding(nn.Module):
    def __init__(self):
        super().__init__()
        self.pos = nn.Embedding(INPUT_SIZE + 1, UNITS)
        nn.init.zeros_(self.pos.weight)
        self.lips = LandmarkEmbedding(40 * 2, LANDMARK_UNITS)
        self.hand = LandmarkEmbedding(21 * 2, LANDMARK_UNITS)
        self.pose = LandmarkEmbedding(5 * 2, LANDMARK_UNITS)
        self.landmark_weights = nn.Parameter(torch.zeros(3))  # learnable attention over landmark groups
        self.fc1 = glorot(nn.Linear(LANDMARK_UNITS, UNITS, bias=False))
        self.fc2 = he_uniform(nn.Linear(UNITS, UNITS, bias=False))

    def forward(self, lips, hand, pose, fidx):
        x = torch.stack([self.lips(lips), self.hand(hand), self.pose(pose)], dim=-1)  # [B,T,U,3]
        x = (x * F.softmax(self.landmark_weights, dim=0).to(x.dtype)).sum(-1)
        x = self.fc2(F.gelu(self.fc1(x)))
        # positional index = relative position of the original frame (padding -> INPUT_SIZE)
        mx = fidx.max(dim=1, keepdim=True).values.clamp(min=1.0)
        pos = (fidx / mx * INPUT_SIZE).long().clamp(0, INPUT_SIZE)
        pos = pos.masked_fill(fidx == -1, INPUT_SIZE)
        return x + self.pos(pos)


class MultiHeadAttention(nn.Module):
    def __init__(self, d, heads):
        super().__init__()
        self.h = heads
        self.wq, self.wk, self.wv, self.wo = (glorot(nn.Linear(d, d)) for _ in range(4))

    def forward(self, x, key_mask):  # key_mask [B,T] bool, True = real frame
        B, T, D = x.shape
        split = lambda t: t.view(B, T, self.h, D // self.h).transpose(1, 2)
        q, k, v = split(self.wq(x)), split(self.wk(x)), split(self.wv(x))
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=key_mask[:, None, None, :])
        return self.wo(out.transpose(1, 2).reshape(B, T, D))


class Block(nn.Module):
    """Shallow transformer block without LayerNorm (as in the TF v5 notebook)."""

    def __init__(self):
        super().__init__()
        self.mha = MultiHeadAttention(UNITS, NUM_HEADS)
        self.mlp = nn.Sequential(
            glorot(nn.Linear(UNITS, UNITS * MLP_RATIO)), nn.GELU(), nn.Dropout(MLP_DROPOUT),
            he_uniform(nn.Linear(UNITS * MLP_RATIO, UNITS)),
        )

    def forward(self, x, key_mask):
        x = x + self.mha(x, key_mask)
        return x + self.mlp(x)


class SignModel(nn.Module):
    def __init__(self, stats=None):
        super().__init__()
        dflt = {"lips": 40, "hand": 21, "pose": 5}
        for k, n in dflt.items():  # normalisation stats live in the state_dict
            self.register_buffer(f"{k}_mean", (stats or {}).get(f"{k}_mean", torch.zeros(n, 2)))
            self.register_buffer(f"{k}_std", (stats or {}).get(f"{k}_std", torch.ones(n, 2)))
        self.embedding = Embedding()
        self.blocks = nn.ModuleList([Block() for _ in range(NUM_BLOCKS)])
        self.dropout = nn.Dropout(CLASSIFIER_DROPOUT)
        self.head = glorot(nn.Linear(UNITS, NUM_CLASSES))

    @staticmethod
    def _norm(v, mean, std):
        return torch.where(v == 0, torch.zeros_like(v), (v - mean) / std).flatten(2)

    def forward(self, frames, fidx):  # frames [B,T,66,3], fidx [B,T]
        mask0 = fidx != -1
        if self.training:  # random frame dropout, always keep >=1 frame
            mask = (torch.rand_like(fidx) > FRAME_MASK_P) & mask0
            mask = torch.where(mask.any(1, keepdim=True), mask, mask0)
        else:
            mask = mask0
        xy = frames[..., :2]
        lips = self._norm(xy[:, :, LIPS_SL], self.lips_mean, self.lips_std)
        hand = self._norm(xy[:, :, HAND_SL], self.hand_mean, self.hand_std)
        pose = self._norm(xy[:, :, POSE_SL], self.pose_mean, self.pose_std)

        x = self.embedding(lips, hand, pose, fidx)
        for blk in self.blocks:
            x = blk(x, mask)
        m = mask.unsqueeze(-1).to(x.dtype)
        x = (x * m).sum(1) / m.sum(1).clamp(min=1.0)  # masked mean pooling
        return self.head(self.dropout(x))  # logits


# ----------------------------------------------------------------------------
# Training
# ----------------------------------------------------------------------------
class BalancedSampler:
    """Each batch contains `n` samples of every class (same as get_train_batch_all_signs)."""

    def __init__(self, y, idxs, n, seed):
        ys = y[idxs]
        order = np.argsort(ys, kind="stable")
        self.pool = idxs[order]
        counts = np.bincount(ys, minlength=NUM_CLASSES)
        self.classes = np.flatnonzero(counts > 0)
        self.counts = counts[self.classes]
        self.starts = np.concatenate([[0], np.cumsum(counts)])[self.classes]
        self.n = n
        self.labels = np.repeat(self.classes, n)
        self.rng = np.random.default_rng(seed)

    def sample(self):
        r = (self.rng.random((len(self.classes), self.n)) * self.counts[:, None]).astype(np.int64)
        return self.pool[self.starts[:, None] + r].reshape(-1), self.labels


def augment(x):
    """x: [B, T, 66, 3] float tensor (0 = missing landmark). Returns an augmented copy; missing stays missing."""
    B, dev = x.shape[0], x.device
    xy = x[..., :2]
    present = (xy != 0).any(-1, keepdim=True)                                  # [B,T,66,1]
    center = (xy * present).sum((1, 2)) / present.sum((1, 2)).clamp(min=1)     # [B,2]

    on = (torch.rand(B, device=dev) < AUG_P).float()                          # 1 = augment this sample
    u = lambda lo, hi, *shape: torch.rand(B, *shape, device=dev) * (hi - lo) + lo
    sx = 1 + on * (u(*AUG_SCALE) - 1)
    sy = 1 + on * (u(*AUG_SCALE) - 1)
    th = on * u(-AUG_ROT_DEG, AUG_ROT_DEG) * math.pi / 180
    shift = on[:, None] * u(-AUG_SHIFT, AUG_SHIFT, 2)
    cos, sin = torch.cos(th), torch.sin(th)
    M = torch.stack([torch.stack([cos * sx, -sin * sy], -1),
                     torch.stack([sin * sx, cos * sy], -1)], -2)               # [B,2,2] = R @ S

    c = center[:, None, None, :]
    new = torch.einsum("bij,btlj->btli", M, xy - c) + c + shift[:, None, None, :]
    new = new + torch.randn_like(new) * AUG_NOISE * on[:, None, None, None]
    new = torch.where(present, new, torch.zeros_like(new))
    out = torch.cat([new, x[..., 2:]], dim=-1)

    drop = (torch.rand(B, device=dev) < AUG_LIPS_DROP_P)[:, None, None, None]
    out[:, :, LIPS_SL] = out[:, :, LIPS_SL] * (~drop)
    return out


def topk_hits(logits, y):
    hit = logits.topk(10, dim=1).indices == y[:, None]
    return hit[:, :1].sum().item(), hit[:, :5].sum().item(), hit.sum().item()


@torch.no_grad()
def evaluate(model, X, FI, Y, idxs, device, amp, bs=1024):
    model.eval()
    loss = c1 = c5 = c10 = 0
    for i in range(0, len(idxs), bs):
        b = idxs[i:i + bs]
        xb, fb, yb = X[b].to(device), FI[b].to(device), Y[b].to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
            logits = model(xb, fb)
        loss += F.cross_entropy(logits.float(), yb, reduction="sum").item()
        h1, h5, h10 = topk_hits(logits, yb)
        c1, c5, c10 = c1 + h1, c5 + h5, c10 + h10
    n = len(idxs)
    return dict(loss=loss / n, acc=c1 / n, top5=c5 / n, top10=c10 / n)


def train():
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    amp = device.type == "cuda" and USE_AMP
    data_dev = device if (DATA_ON_GPU and device.type == "cuda") else torch.device("cpu")

    X_np = np.load(f"{CACHE_DIR}/X.npy")
    FI_np = np.load(f"{CACHE_DIR}/fidx.npy")
    y_np = np.load(f"{CACHE_DIR}/y.npy")
    with open(f"{CACHE_DIR}/sign2idx.json") as f:
        sign2idx = json.load(f)
    print(f"X {X_np.shape} | device {device} | amp {amp} | data on {data_dev}")

    all_idx = np.arange(len(y_np))
    if USE_VAL:
        from sklearn.model_selection import GroupShuffleSplit
        pid = np.load(f"{CACHE_DIR}/participant.npy")
        tr_idx, va_idx = next(GroupShuffleSplit(test_size=0.10, n_splits=2, random_state=SEED)
                              .split(all_idx, y_np, groups=pid))
        tr_idx, va_idx = np.sort(tr_idx), np.sort(va_idx)
        print(f"train {len(tr_idx)} | val {len(va_idx)} | participants overlap: "
              f"{len(set(pid[tr_idx]) & set(pid[va_idx]))}")
    else:
        tr_idx, va_idx = all_idx, None

    stats = compute_stats(X_np, tr_idx)
    X = torch.from_numpy(X_np).to(data_dev)
    FI = torch.from_numpy(FI_np).to(data_dev)
    Y = torch.from_numpy(y_np).to(data_dev)
    va_t = torch.from_numpy(va_idx).to(data_dev) if va_idx is not None else None

    model = SignModel(stats).to(device)
    print(f"params: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")
    # torch AdamW decays by lr*wd per step == TF's tfa AdamW with weight_decay = lr * WD_RATIO
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD_RATIO)
    scaler = torch.cuda.amp.GradScaler(enabled=amp)
    sampler = BalancedSampler(y_np, tr_idx, ALL_SIGNS_N, SEED)
    steps = max(1, len(tr_idx) // (len(sampler.classes) * ALL_SIGNS_N))
    history, best = [], -1.0

    for epoch in range(EPOCHS):
        lr = 0.5 * (1 + math.cos(math.pi * epoch / EPOCHS)) * LR  # cosine decay, no warmup
        for g in opt.param_groups:
            g["lr"] = lr
        model.train()
        t0, tot_loss, n_seen, c1, c5, c10 = time.time(), 0.0, 0, 0, 0, 0
        for _ in range(steps):
            idx, labels = sampler.sample()
            idx = torch.from_numpy(idx).to(data_dev)
            xb, fb = X[idx].to(device, non_blocking=True), FI[idx].to(device, non_blocking=True)
            yb = torch.from_numpy(labels).to(device)
            if AUGMENT:
                xb = augment(xb)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
                logits = model(xb, fb)
            loss = F.cross_entropy(logits.float(), yb, label_smoothing=LABEL_SMOOTHING)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()

            tot_loss += loss.item() * len(yb)
            n_seen += len(yb)
            h = topk_hits(logits.detach(), yb)
            c1, c5, c10 = c1 + h[0], c5 + h[1], c10 + h[2]

        rec = dict(epoch=epoch + 1, lr=lr, loss=tot_loss / n_seen, acc=c1 / n_seen,
                   top5=c5 / n_seen, top10=c10 / n_seen)
        msg = (f"ep {epoch + 1:3d}/{EPOCHS} lr {lr:.2e} | loss {rec['loss']:.4f} acc {rec['acc']:.4f} "
               f"top5 {rec['top5']:.4f} top10 {rec['top10']:.4f}")
        if va_t is not None:
            v = evaluate(model, X, FI, Y, va_t, device, amp)
            rec.update({f"val_{k}": val for k, val in v.items()})
            msg += f" | val loss {v['loss']:.4f} acc {v['acc']:.4f} top5 {v['top5']:.4f} top10 {v['top10']:.4f}"
            if v["acc"] > best:
                best = v["acc"]
                save_ckpt(model, sign2idx, OUT_PATH.replace(".pt", "_best.pt"))
        print(msg + f" | {time.time() - t0:.0f}s", flush=True)
        history.append(rec)

    save_ckpt(model, sign2idx, OUT_PATH)
    with open(OUT_PATH.replace(".pt", "_history.json"), "w") as f:
        json.dump(history, f, indent=1)
    w = F.softmax(model.embedding.landmark_weights.detach().cpu(), 0).tolist()
    print(f"saved {OUT_PATH} | landmark weights lips/hand/pose: {[round(x * 100, 1) for x in w]}%")


def save_ckpt(model, sign2idx, path):
    torch.save({"model": model.state_dict(), "sign2idx": sign2idx}, path)


# ----------------------------------------------------------------------------
# Inference on one parquet file
# ----------------------------------------------------------------------------
@torch.no_grad()
def predict(parquet_path, ckpt=OUT_PATH):
    ck = torch.load(ckpt, map_location="cpu")
    model = SignModel()
    model.load_state_dict(ck["model"])
    model.eval()
    idx2sign = {v: k for k, v in ck["sign2idx"].items()}
    data, fidx = preprocess_sequence(load_parquet(parquet_path))
    logits = model(torch.from_numpy(data)[None], torch.from_numpy(fidx)[None])
    probs = F.softmax(logits, dim=1)[0]
    for p, i in zip(*probs.topk(5)):
        print(f"{idx2sign[i.item()]:<15s} {p.item():.3f}")


# ----------------------------------------------------------------------------
# Run
# ----------------------------------------------------------------------------
if __name__ == "__main__":
    # the sign2idx.json file is written last, so it marks a finished preprocessing run
    if not os.path.exists(f"{CACHE_DIR}/sign2idx.json"):
        preprocess_all()
    train()